# Thesis Figures — GHI Probabilistic Forecasting Pipeline

Publication-quality figures for Harry's thesis. Based on:
- **V1 Fixed**: Baseline XGBoost (single model, lr=0.05, depth=8)
- **V2 Fixed**: Tuned XGBoost + 5-seed ensemble (lr=0.02, depth=6)

Both use CQR-XGBQ with 19 calibrated quantiles (P05–P95).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 14, 'axes.labelsize': 12,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

# Colors
C_OURS = '#2171b5'
C_PIETER = '#e6550d'
C_NWP = '#2171b5'
C_NO_NWP = '#d94701'
C_V1 = '#6baed6'
C_V2 = '#2171b5'

# Paths
OUT = Path('../pipeline_outputs')
FIG = OUT / 'figures'
FIG.mkdir(exist_ok=True)

# Load data
forecast = pd.read_parquet(OUT / 'forecast_ghi_quantiles_daily.parquet')
test = forecast[forecast['split_name'] == 'test'].copy()
test['hour'] = pd.to_datetime(test['target_time_local']).dt.hour
test['month'] = pd.to_datetime(test['target_time_local']).dt.month
test['date'] = pd.to_datetime(test['target_day_local']).dt.date

dataset = pd.read_parquet(OUT / 'dataset_issue_target.parquet')

# Also load raw (pre-CQR) if available
raw_path = OUT / 'forecast_ghi_quantiles_daily_base_raw.parquet'
has_raw = raw_path.exists()
if has_raw:
    forecast_raw = pd.read_parquet(raw_path)
    test_raw = forecast_raw[forecast_raw['split_name'] == 'test'].copy()

quantiles = [round(0.05 + 0.05 * i, 2) for i in range(19)]
q_cols = [f'q{q:.2f}' for q in quantiles]

print(f'Test set: {len(test)} hours, {test["date"].nunique()} days')
print(f'Raw forecast available: {has_raw}')
print(f'Figures will be saved to: {FIG.resolve()}')

## Figure 1: Quantile Fan Chart — Representative Days

Shows P05–P95 prediction intervals for a clear, cloudy, and mixed day.

In [ ]:
# Find representative days based on daily GHI characteristics
daily = test.groupby('date').agg(
    ghi_total=('label_ghi_obs_wm2', 'sum'),
    ghi_std=('label_ghi_obs_wm2', 'std'),
    ghi_max=('label_ghi_obs_wm2', 'max'),
).reset_index()

# Clear day: high total, low std relative to total
daily['cv'] = daily['ghi_std'] / (daily['ghi_total'] + 1)
clear_day = daily.nlargest(20, 'ghi_total').nsmallest(1, 'cv')['date'].values[0]

# Cloudy day: low total but not zero
cloudy_candidates = daily[(daily['ghi_total'] > 500) & (daily['ghi_total'] < daily['ghi_total'].quantile(0.3))]
cloudy_day = cloudy_candidates.nlargest(1, 'cv')['date'].values[0]

# Mixed day: medium total, high variability
mixed_candidates = daily[(daily['ghi_total'] > daily['ghi_total'].quantile(0.3)) &
                         (daily['ghi_total'] < daily['ghi_total'].quantile(0.7))]
mixed_day = mixed_candidates.nlargest(1, 'cv')['date'].values[0]

days = [clear_day, mixed_day, cloudy_day]
labels = ['Clear Day', 'Mixed Day', 'Cloudy Day']

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)

band_pairs = [('q0.05', 'q0.95', 0.15), ('q0.10', 'q0.90', 0.2),
              ('q0.15', 'q0.85', 0.25), ('q0.20', 'q0.80', 0.3),
              ('q0.25', 'q0.75', 0.4)]

for ax, day, label in zip(axes, days, labels):
    d = test[test['date'] == day].sort_values('hour')
    hours = d['hour'].values
    actual = d['label_ghi_obs_wm2'].values

    for q_lo, q_hi, alpha in band_pairs:
        ax.fill_between(hours, d[q_lo].values, d[q_hi].values,
                        alpha=alpha, color=C_OURS, linewidth=0)

    ax.plot(hours, d['q0.50'].values, '--', color=C_OURS, linewidth=1.5, label='P50 (median)')
    ax.plot(hours, actual, 'o-', color='#c0392b', markersize=4, linewidth=1.5, label='Actual GHI')

    ax.set_title(f'{label}\n({day})', fontsize=12)
    ax.set_xlabel('Hour of Day')
    ax.set_xlim(4, 20)
    ax.set_xticks(range(4, 21, 2))

axes[0].set_ylabel('GHI (W/m²)')
axes[0].legend(loc='upper left', fontsize=9)

# Add band legend manually
from matplotlib.patches import Patch
band_legend = [Patch(facecolor=C_OURS, alpha=0.4, label='P25–P75'),
               Patch(facecolor=C_OURS, alpha=0.2, label='P10–P90'),
               Patch(facecolor=C_OURS, alpha=0.15, label='P05–P95')]
axes[2].legend(handles=band_legend, loc='upper right', fontsize=9)

fig.suptitle('CQR-XGBQ Probabilistic GHI Forecast — Representative Days', fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig(FIG / 'fig1_quantile_fan_chart.png')
fig.savefig(FIG / 'fig1_quantile_fan_chart.pdf')
plt.show()
print('Saved: fig1_quantile_fan_chart')

## Figure 2: Reliability Diagram (Calibration Plot)

Nominal vs actual coverage — proves CQR calibration works. Shows both raw XGBQ and post-CQR.

In [ ]:
# Reliability diagram: nominal quantile vs empirical coverage (daylight only)
daylight = test['solar_elevation'].values > 0
y_day = test.loc[daylight, 'label_ghi_obs_wm2'].values

# Post-CQR coverage
cqr_coverage = []
for q, col in zip(quantiles, q_cols):
    pred = test.loc[daylight, col].values
    cov = np.mean(y_day <= pred)
    cqr_coverage.append(cov)

fig, ax = plt.subplots(figsize=(6, 6))

# Raw coverage if available
if has_raw:
    daylight_raw = test_raw['solar_elevation'].values > 0
    y_day_raw = test_raw.loc[daylight_raw, 'label_ghi_obs_wm2'].values
    raw_coverage = []
    for q, col in zip(quantiles, q_cols):
        if col in test_raw.columns:
            pred_raw = test_raw.loc[daylight_raw, col].values
            raw_coverage.append(np.mean(y_day_raw <= pred_raw))
        else:
            raw_coverage.append(np.nan)
    ax.plot(quantiles, raw_coverage, 's--', color=C_NO_NWP, markersize=6,
            label='Raw XGBQ (before CQR)', alpha=0.8)

ax.plot(quantiles, cqr_coverage, 'o-', color=C_OURS, markersize=7,
        label='After CQR Calibration', linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Perfect calibration')

ax.set_xlabel('Nominal Quantile Level')
ax.set_ylabel('Empirical Coverage (fraction below)')
ax.set_title('Reliability Diagram — CQR Calibration Effect\n(Daylight Test Hours)')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect('equal')
ax.legend(loc='upper left', fontsize=10)

# Add shaded ±5% tolerance band
ax.fill_between([0, 1], [0-0.05, 1-0.05], [0+0.05, 1+0.05],
                alpha=0.1, color='gray', label='_nolegend_')

fig.savefig(FIG / 'fig2_reliability_diagram.png')
fig.savefig(FIG / 'fig2_reliability_diagram.pdf')
plt.show()
print('Saved: fig2_reliability_diagram')

## Figure 3: NWP Ablation — With vs Without GFS Features

V1 (baseline) and V2 (tuned+5-seed) results from the NWP contribution experiment.

In [ ]:
# NWP ablation results (from nwp_contribution notebook)
nwp_data = {
    'V1 Baseline': {'with': {'MAE': 84.04, 'R2': 0.8046, 'CRPS': 31.64},
                     'without': {'MAE': 132.31, 'R2': 0.5759, 'CRPS': 47.73}},
    'V2 Tuned+5seed': {'with': {'MAE': 83.16, 'R2': 0.8067, 'CRPS': 31.24},
                        'without': {'MAE': 130.57, 'R2': 0.5785, 'CRPS': 47.07}},
}

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metrics = [('MAE', 'MAE (W/m²)', True), ('R2', 'R²', False), ('CRPS', 'CRPS', True)]

x = np.arange(2)
width = 0.32

for ax, (metric, ylabel, lower_better) in zip(axes, metrics):
    with_vals = [nwp_data[v]['with'][metric] for v in nwp_data]
    without_vals = [nwp_data[v]['without'][metric] for v in nwp_data]

    bars1 = ax.bar(x - width/2, with_vals, width, label='With NWP', color=C_NWP, alpha=0.85)
    bars2 = ax.bar(x + width/2, without_vals, width, label='Without NWP', color=C_NO_NWP, alpha=0.85)

    ax.set_ylabel(ylabel)
    ax.set_xticks(x)
    ax.set_xticklabels(['V1\nBaseline', 'V2\nTuned+5seed'])

    # Add improvement annotations
    for i in range(2):
        w = with_vals[i]
        wo = without_vals[i]
        if metric == 'R2':
            delta = w - wo
            txt = f'+{delta:.2f}'
        else:
            pct = (wo - w) / wo * 100
            txt = f'-{pct:.0f}%'
        y_pos = max(w, wo) * 1.02
        ax.annotate(txt, xy=(x[i], y_pos), ha='center', fontsize=10, fontweight='bold', color='#333')

    ax.legend(fontsize=9)
    if metric == 'R2':
        ax.set_ylim(0, 1.05)

fig.suptitle('NWP Contribution: Impact of GFS Forecast Features on GHI Prediction\n(Daylight Hours, Test Set)', fontsize=13)
plt.tight_layout()
fig.savefig(FIG / 'fig3_nwp_ablation.png')
fig.savefig(FIG / 'fig3_nwp_ablation.pdf')
plt.show()
print('Saved: fig3_nwp_ablation')

## Figure 4: Pieter Comparison — Per-Season MAE (All Hours)

In [ ]:
# Pieter comparison (per-season, all hours)
seasons = ['Summer', 'Fall', 'Winter', 'Spring']
mae_ours = [54.71, 30.68, 32.39, 47.56]
mae_pieter = [105.13, 93.57, 99.81, 118.77]

fig, ax = plt.subplots(figsize=(9, 5.5))
x = np.arange(len(seasons))
width = 0.35

bars1 = ax.bar(x - width/2, mae_ours, width, label='CQR-XGBQ (Ours)', color=C_OURS, alpha=0.85)
bars2 = ax.bar(x + width/2, mae_pieter, width, label='RNN-LSTM (Pieter, 2023)', color=C_PIETER, alpha=0.85)

# Add improvement labels
for i in range(len(seasons)):
    pct = (mae_pieter[i] - mae_ours[i]) / mae_pieter[i] * 100
    y_pos = mae_pieter[i] + 2
    ax.annotate(f'-{pct:.0f}%', xy=(x[i], y_pos), ha='center', fontsize=11,
                fontweight='bold', color='#2c3e50')

ax.set_ylabel('MAE (W/m²)')
ax.set_xlabel('Season')
ax.set_title('Per-Season GHI Forecast MAE Comparison (All Hours)\nCQR-XGBQ vs RNN-LSTM')
ax.set_xticks(x)
ax.set_xticklabels(seasons)
ax.legend(fontsize=11)
ax.set_ylim(0, 140)

plt.tight_layout()
fig.savefig(FIG / 'fig4_pieter_comparison.png')
fig.savefig(FIG / 'fig4_pieter_comparison.pdf')
plt.show()
print('Saved: fig4_pieter_comparison')

## Figure 5: Feature Importance (Top 20)

XGBoost gain-based importance. NWP features highlighted in blue, non-NWP in gray.

In [ ]:
import xgboost as xgb

NWP_FEATURES = {'dswrf', 'tcc', 'lcc', 'mcc', 'hcc', 't2m', 't2m_c', 'u', 'v',
                'r2', 'tp', 'wind_speed', 'nwp_clear_ratio', 'dswrf_lag24'}

# Get feature columns (same logic as pipeline)
drop_always = {'label_ghi_obs_wm2', 'ghi_obs_wm2', 'Global_Solar_Radiation_MJm2',
               'issue_day_local', 'target_day_local', 'target_time_local',
               'issue_time_local', 'ts_utc', 'chosen_init_utc', 'deadline_utc',
               'split_name', 'YYYYMM', 'DD', 'HH', 'Station_No'}
drop_obs = {'Sunshine_Duration_hour', 'Total_Cloud_Amount_tenths', 'Air_Temperature_C',
            'Relative_Humidity_percent', 'Wind_Speed_ms', 'Wind_Direction_degree',
            'Gust_Speed_ms', 'Gust_Direction_degree', 'Precipitation_mm',
            'Visibility_km', 'Station_Pressure_hPa', 'Sea_Level_Pressure_hPa'}
drop_set = drop_always | drop_obs

feature_cols = [c for c in dataset.columns if c not in drop_set and pd.api.types.is_numeric_dtype(dataset[c])]

# Train a single Q0.50 model with V2 params
train_mask = (dataset['split_name'] == 'train') & (dataset['solar_elevation'] > 0)
valid_mask = (dataset['split_name'] == 'valid') & (dataset['solar_elevation'] > 0)

X_train = dataset.loc[train_mask, feature_cols].fillna(0)
y_train = dataset.loc[train_mask, 'label_ghi_obs_wm2'].fillna(0)
X_valid = dataset.loc[valid_mask, feature_cols].fillna(0)
y_valid = dataset.loc[valid_mask, 'label_ghi_obs_wm2'].fillna(0)

dtrain = xgb.DMatrix(X_train, label=y_train)
dvalid = xgb.DMatrix(X_valid, label=y_valid)

params = {'objective': 'reg:quantileerror', 'quantile_alpha': 0.5,
          'tree_method': 'hist', 'learning_rate': 0.02, 'max_depth': 6,
          'subsample': 0.65, 'colsample_bytree': 0.7, 'min_child_weight': 5}

model = xgb.train(params, dtrain, num_boost_round=3000,
                  evals=[(dvalid, 'valid')], early_stopping_rounds=50, verbose_eval=False)

# Get importance
importance = model.get_score(importance_type='gain')
imp_df = pd.DataFrame({'feature': list(importance.keys()), 'gain': list(importance.values())})
imp_df = imp_df.sort_values('gain', ascending=True).tail(20)
imp_df['is_nwp'] = imp_df['feature'].isin(NWP_FEATURES)

fig, ax = plt.subplots(figsize=(8, 7))
colors = [C_NWP if nwp else '#999999' for nwp in imp_df['is_nwp']]
ax.barh(range(len(imp_df)), imp_df['gain'].values, color=colors, alpha=0.85)
ax.set_yticks(range(len(imp_df)))
ax.set_yticklabels(imp_df['feature'].values, fontsize=10)
ax.set_xlabel('Gain (Feature Importance)')
ax.set_title('Top 20 Feature Importance — V2 XGBoost Q0.50\n(Blue = NWP/GFS, Gray = Non-NWP)')

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=C_NWP, label='NWP (GFS forecast)'),
                   Patch(color='#999999', label='Non-NWP (solar/calendar/lagged)')],
          loc='lower right', fontsize=10)

plt.tight_layout()
fig.savefig(FIG / 'fig5_feature_importance.png')
fig.savefig(FIG / 'fig5_feature_importance.pdf')
plt.show()
print('Saved: fig5_feature_importance')

## Figure 6: Seasonal Error Heatmap (Hour x Month)

In [ ]:
# Heatmap: MAE by hour of day x month (daylight only)
test['abs_error'] = np.abs(test['label_ghi_obs_wm2'] - test['q0.50'])

# Compute mean MAE per (hour, month)
pivot = test[test['solar_elevation'] > 0].groupby(['hour', 'month'])['abs_error'].mean().unstack(fill_value=np.nan)

# Create full grid and mask nighttime
full_grid = pd.DataFrame(index=range(24), columns=range(1, 13), dtype=float)
for h in range(24):
    for m in range(1, 13):
        if h in pivot.index and m in pivot.columns:
            full_grid.loc[h, m] = pivot.loc[h, m]

fig, ax = plt.subplots(figsize=(10, 6))
data = full_grid.values.astype(float)
masked = np.ma.masked_where(np.isnan(data), data)

im = ax.imshow(masked, aspect='auto', cmap='YlOrRd', origin='lower',
               extent=[0.5, 12.5, -0.5, 23.5], interpolation='nearest')

ax.set_xlabel('Month')
ax.set_ylabel('Hour of Day')
ax.set_title('GHI Forecast MAE by Hour and Month\n(Daylight Hours, Test Set — V2 Fixed)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                     'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], fontsize=9)
ax.set_yticks(range(0, 24, 2))

cbar = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label('MAE (W/m²)')

plt.tight_layout()
fig.savefig(FIG / 'fig6_error_heatmap.png')
fig.savefig(FIG / 'fig6_error_heatmap.pdf')
plt.show()
print('Saved: fig6_error_heatmap')

## Figure 7: Scatter Plot — Predicted vs Actual GHI (Daylight)

In [ ]:
# Scatter: Predicted vs Actual (daylight only)
day_mask = test['solar_elevation'].values > 0
y_actual = test.loc[day_mask, 'label_ghi_obs_wm2'].values
y_pred = test.loc[day_mask, 'q0.50'].values

# Compute metrics
mae = np.mean(np.abs(y_actual - y_pred))
rmse = np.sqrt(np.mean((y_actual - y_pred) ** 2))
ss_res = np.sum((y_actual - y_pred) ** 2)
ss_tot = np.sum((y_actual - np.mean(y_actual)) ** 2)
r2 = 1 - ss_res / ss_tot

fig, ax = plt.subplots(figsize=(7, 7))

# Hexbin for density
hb = ax.hexbin(y_actual, y_pred, gridsize=50, cmap='Blues', mincnt=1, linewidths=0.2)
cb = fig.colorbar(hb, ax=ax, shrink=0.8, pad=0.02)
cb.set_label('Count')

# 1:1 line
lim = max(y_actual.max(), y_pred.max()) * 1.05
ax.plot([0, lim], [0, lim], 'r--', linewidth=1.5, alpha=0.7, label='1:1 line')

# Annotation box
textstr = f'R² = {r2:.4f}\nMAE = {mae:.1f} W/m²\nRMSE = {rmse:.1f} W/m²\nn = {len(y_actual)}'
props = dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray')
ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', bbox=props)

ax.set_xlabel('Actual GHI (W/m²)')
ax.set_ylabel('Predicted GHI — P50 (W/m²)')
ax.set_title('Predicted vs Actual GHI\n(V2 Fixed — Daylight Test Hours)')
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_aspect('equal')
ax.legend(loc='lower right', fontsize=10)

plt.tight_layout()
fig.savefig(FIG / 'fig7_scatter_pred_vs_actual.png')
fig.savefig(FIG / 'fig7_scatter_pred_vs_actual.pdf')
plt.show()
print('Saved: fig7_scatter_pred_vs_actual')

## Summary

All 7 figures saved to `pipeline_outputs/figures/` as PNG (300 dpi) and PDF:

| Figure | File | Purpose |
|--------|------|---------|
| 1 | `fig1_quantile_fan_chart` | Probabilistic forecast visualization (clear/mixed/cloudy) |
| 2 | `fig2_reliability_diagram` | CQR calibration proof (nominal vs actual coverage) |
| 3 | `fig3_nwp_ablation` | NWP contribution (V1 & V2, with/without GFS) |
| 4 | `fig4_pieter_comparison` | Per-season MAE vs Pieter's RNN-LSTM |
| 5 | `fig5_feature_importance` | Top 20 features (NWP vs non-NWP highlighted) |
| 6 | `fig6_error_heatmap` | Seasonal error patterns (hour x month) |
| 7 | `fig7_scatter_pred_vs_actual` | Predicted vs actual with R² annotation |